# Regression with a Single-country Example

This notebook uses the United States as a time-series-style example to illustrate descriptive statistics, variable transformation, correlation analysis, and simple OLS regression.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "04_regression_single_country"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "wdi" / "WDI_course_subset.csv")

# Use the United States as the example country.
df_US = df[df["Country Name"] == "United States"]

WDI_US = df_US.drop(columns="Indicator Code").melt(
    id_vars=["Country Name", "Country Code", "Indicator Name"],
    var_name="Year",
).pivot_table(
    values="value",
    index=["Country Name", "Country Code", "Year"],
    columns="Indicator Name",
).reset_index().rename_axis("", axis=1)

# Convert Year from string-like values to integers.
WDI_US["Year"] = WDI_US["Year"].astype(str).astype(int)

isna_data = WDI_US.isna().sum().sort_values(ascending=True)
isna_data.to_csv(MODULE_OUTPUT_DIR / "isna_data_US.csv", index=True)

WDI_US.head()

# Q1. How can we choose variables for a research question and describe them statistically?

In [ ]:
# `describe()` returns count, mean, standard deviation, minimum, maximum, and quartiles.
WDI_US[
    [
        "GDP per capita (constant 2015 US$)",
        "Gross fixed capital formation (% of GDP)",
        "Trade (% of GDP)",
        "Inflation, consumer prices (annual %)",
        "Foreign direct investment, net inflows (% of GDP)",
    ]
].describe()



# Q2. How can we do simple transformations before regression?

In [ ]:
import numpy as np
pd.options.mode.chained_assignment = None

# Build a simple growth-regression dataset.
WDI_US["lngdpc"] = np.log(WDI_US["GDP per capita (constant 2015 US$)"])
WDI_US["capital_share"] = WDI_US["Gross fixed capital formation (% of GDP)"]
WDI_US["trade_open"] = WDI_US["Trade (% of GDP)"]
WDI_US["inflation"] = WDI_US["Inflation, consumer prices (annual %)"]
WDI_US["fdi"] = WDI_US["Foreign direct investment, net inflows (% of GDP)"]

WDI_US.to_csv(MODULE_OUTPUT_DIR / "WDI_US_transformed.csv", index=False)

WDI_US.describe()



## 2.1 Select complete years and export a transposed descriptive statistics table

In [ ]:
# The variables below are reasonably complete after 1990, so we trim the sample period.
ols_selected = WDI_US[(WDI_US["Year"] < 2021) & (WDI_US["Year"] > 1990)]

summary_table = ols_selected[
    ["lngdpc", "capital_share", "trade_open", "inflation", "fdi"]
].describe().T

summary_table.to_csv(MODULE_OUTPUT_DIR / "summary_table.csv", index=True)
summary_table



## 2.2 How can we compute a correlation matrix with significance stars?

In [ ]:
cor_data = ols_selected[
    ["lngdpc", "capital_share", "trade_open", "inflation", "fdi"]
]

from scipy.stats import pearsonr

rho = cor_data.corr()
pval = cor_data.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*rho.shape)
p = pval.map(lambda x: "".join(["*" for t in [0.05, 0.01, 0.001] if x <= t]))
correlation = rho.round(4).astype(str) + p

correlation.to_csv(MODULE_OUTPUT_DIR / "correlation.csv", index=True)
correlation

### If you want to export a table to Word format, see:
https://rowannicholls.github.io/python/data/export_to_word.html

## 2.3 The same correlation matrix can also be visualized as a heat map.

In [ ]:
%pip install -q seaborn

In [ ]:
import seaborn as sb
import matplotlib.pyplot as plt

sb.heatmap(rho, cmap="Greens", annot=True)
plt.savefig(MODULE_OUTPUT_DIR / "heatmap.png", bbox_inches="tight")

# Q3. How can we run a simple OLS regression?

In [ ]:
%pip install -q statsmodels openpyxl

## 3.1 Run a basic single-equation OLS model

In [ ]:
import statsmodels.api as sm

# Define predictor and response variables.
y = ols_selected["lngdpc"]
x = ols_selected["capital_share"]

# Add a constant and fit the model.
x = sm.add_constant(x)
model = sm.OLS(y, x).fit()

print(model.summary())



## 3.2 Export a regression table with significance stars

In [ ]:
from statsmodels.iolib.summary2 import summary_col

# Define the dependent variable.
y = ols_selected[["lngdpc"]]

# Define the independent variables.
X1 = ols_selected[["capital_share"]]
X2 = ols_selected[["capital_share", "trade_open"]]
X3 = ols_selected[["capital_share", "trade_open", "inflation"]]
X4 = ols_selected[["capital_share", "trade_open", "inflation", "fdi"]]

# Add constant terms.
X1 = sm.add_constant(X1)
X2 = sm.add_constant(X2)
X3 = sm.add_constant(X3)
X4 = sm.add_constant(X4)

# Fit OLS models.
model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()
model3 = sm.OLS(y, X3).fit()
model4 = sm.OLS(y, X4).fit()

# Create a summary table with significance stars.
reg = summary_col(
    [model1, model2, model3, model4],
    stars=True,
    model_names=["Model 1", "Model 2", "Model 3", "Model 4"],
    float_format="%0.2f",
    info_dict={"N": lambda result: result.nobs},
)

reg_df = reg.tables[0].reset_index(drop=False)
reg_df.to_excel(MODULE_OUTPUT_DIR / "regression_results.xlsx", index=False)

reg_df



## 3.3 How can we draw a fitted line?

In [ ]:
# Find the best-fitting line.
a, b = np.polyfit(ols_selected["capital_share"], ols_selected["lngdpc"], 1)

# Add the scatter points.
plt.scatter(ols_selected["capital_share"], ols_selected["lngdpc"], color="purple")

# Add the fitted line.
plt.plot(ols_selected["capital_share"], a * ols_selected["capital_share"] + b)

# Add the fitted equation.
plt.text(20, 11, "y = " + "{:.3f}".format(b) + " + {:.3f}".format(a) + "x")

# Add axis labels.
plt.xlabel("Capital formation share of GDP")
plt.ylabel("Log GDP per capita")
plt.savefig(MODULE_OUTPUT_DIR / "fitted_line.png", bbox_inches="tight")

